# Unsupervised Learning: Clustering
### Interactive Notebook for AI/ML Interview Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
np.random.seed(42)
print('Libraries loaded!')

---
## 1. K-Means from Scratch

In [ ]:
def kmeans_scratch(X, k, max_iter=100):
    # Random initialization
    centroids = X[np.random.choice(len(X), k, replace=False)]
    for _ in range(max_iter):
        # Assign clusters
        dists = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(dists, axis=1)
        # Update centroids
        new_centroids = np.array([X[labels==i].mean(axis=0) for i in range(k)])
        if np.allclose(centroids, new_centroids): break
        centroids = new_centroids
    return labels, centroids

X, y_true = make_blobs(n_samples=300, centers=3, random_state=42)
labels, centroids = kmeans_scratch(X, 3)

plt.figure(figsize=(8, 5))
plt.scatter(X[:,0], X[:,1], c=labels, cmap='viridis', s=20, alpha=0.6)
plt.scatter(centroids[:,0], centroids[:,1], c='red', marker='X', s=200, edgecolors='black')
plt.title('K-Means from Scratch'); plt.tight_layout(); plt.show()

---
## 2. Elbow Method & Silhouette

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-'); axes[0].set_title('Elbow Method'); axes[0].set_xlabel('k')
axes[1].plot(K_range, sil_scores, 'ro-'); axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()
print(f'Best k by silhouette: {list(K_range)[np.argmax(sil_scores)]}')

---
## 3. DBSCAN vs K-Means on Non-Spherical Data

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moons)
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=km.labels_, cmap='viridis', s=15)
axes[0].set_title('K-Means (fails on moons!)')
db = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=db.labels_, cmap='viridis', s=15)
axes[1].set_title('DBSCAN (handles non-spherical shapes)')
plt.tight_layout(); plt.show()

---
## 4. Hierarchical Clustering

In [ ]:
X_small, _ = make_blobs(n_samples=30, centers=3, random_state=42)
Z = linkage(X_small, method='ward')
plt.figure(figsize=(10, 4))
dendrogram(Z, leaf_rotation=90)
plt.title('Hierarchical Clustering Dendrogram (Ward)')
plt.xlabel('Sample'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

---
## 5. GMM vs K-Means

In [ ]:
# GMM handles elliptical clusters
from sklearn.datasets import make_blobs
X_ell, _ = make_blobs(n_samples=300, centers=3, random_state=42)
X_ell[:, 1] *= 3  # stretch to make elliptical

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_ell)
axes[0].scatter(X_ell[:,0], X_ell[:,1], c=km.labels_, cmap='viridis', s=15)
axes[0].set_title('K-Means (spherical assumption)')
gmm = GaussianMixture(n_components=3, random_state=42).fit(X_ell)
axes[1].scatter(X_ell[:,0], X_ell[:,1], c=gmm.predict(X_ell), cmap='viridis', s=15)
axes[1].set_title('GMM (handles elliptical)')
plt.tight_layout(); plt.show()

---
## Key Interview Takeaways

1. **K-Means** — fast, simple, but assumes spherical clusters; sensitive to initialization
2. **DBSCAN** — no need to specify k; finds arbitrary shapes; handles noise
3. **GMM** — probabilistic; soft assignments; handles elliptical clusters
4. **Elbow + Silhouette** — two complementary methods to choose k
5. **Hierarchical** — dendrogram shows cluster hierarchy; no need to pre-specify k